In [37]:
import os
import subprocess
from IPython.core.magic import register_cell_magic

# 1. Get the directory where the notebook 
notebook_dir = os.getcwd()

# 2. Go up one level to the project root and into /scripts
script_dir = os.path.abspath(os.path.join(notebook_dir, '..', 'scripts'))

# 3. Create it if it doesn't exist
os.makedirs(script_dir, exist_ok=True)

@register_cell_magic
def form5(line, cell):
    filename = line if line else "temp"
    # Explicitly join with a slash for the .frm file
    filepath = os.path.join(script_dir, f"{filename}.frm")
    
    with open(filepath, 'w') as f:
        f.write(cell)
    
    # Run FORM 5.0
    result = subprocess.run(['form', filepath], capture_output=True, text=True)
    
    # If there is an error, print it clearly
    if result.stderr:
        print("FORM Error:\n", result.stderr)
        return None
    
    # Print to stdout so it's visible in the notebook cell
    print(result.stdout)
    
    # Return the string so it can be captured: output = %%form5 ...
    return result.stdout


In [38]:
%%form5 ee_to_mumu
* Process: e+ e- -> mu+ mu-
Indices mu, nu, mu1, nu1;
Symbols s, t, u, e, pi, alpha, costh;
Vectors p1, p2, p3, p4;

* 1. Amplitude Squared (Diagrammatica Trace Rules)
Local Msq = (e^4 / s^2) * d_(mu, mu1) * d_(nu, nu1) * (g_(1, p2) * g_(1, mu) * g_(1, p1) * g_(1, nu)) *
    (g_(2, p3) * g_(2, mu1) * g_(2, p4) * g_(2, nu1));

Local dSigma = (1 / (64 * pi^2 * s)) * Msq;

trace4, 1;
trace4, 2;
.sort 
Print Msq; 
.sort
contract;
.sort
Print Msq;
.sort

* 2. Physics & Normalization
id e^4 = 16 * pi^2 * alpha^2;
multiply 1/4; * Spin averaging (1/2 * 1/2)

* 3. Kinematics (Massless Limit)
id p1.p1 = 0; id p2.p2 = 0; id p3.p3 = 0; id p4.p4 = 0;
id p1.p2 = s/2; id p3.p4 = s/2;
id p1.p3 = -t/2; id p2.p4 = -t/2;
id p1.p4 = -u/2; id p2.p3 = -u/2;

* 4. Angular dependency
id t = -s/2 * (1 - costh);
id u = -s/2 * (1 + costh);

bracket alpha, s;
Print dSigma;
.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Wed Apr  1 16:53:28 2026
    * Process: e+ e- -> mu+ mu-
    Indices mu, nu, mu1, nu1;
    Symbols s, t, u, e, pi, alpha, costh;
    Vectors p1, p2, p3, p4;
    
    * 1. Amplitude Squared (Diagrammatica Trace Rules)
    Local Msq = (e^4 / s^2) * d_(mu, mu1) * d_(nu, nu1) * (g_(1, p2) * g_(1, mu) * g
    _(1, p1) * g_(1, nu)) *
        (g_(2, p3) * g_(2, mu1) * g_(2, p4) * g_(2, nu1));
    
    Local dSigma = (1 / (64 * pi^2 * s)) * Msq;
    
    trace4, 1;
    trace4, 2;
    .sort 

Time =       0.00 sec    Generated terms =          2
             Msq         Terms in output =          2
                         Bytes used      =        140

Time =       0.00 sec    Generated terms =          2
          dSigma         Terms in output =          2
                         Bytes used      =        156
    Print Msq; 
    .sort

Time =       0.00 sec    Generated terms =          2
             Msq         Terms in output =         

'FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Wed Apr  1 16:53:28 2026\n    * Process: e+ e- -> mu+ mu-\n    Indices mu, nu, mu1, nu1;\n    Symbols s, t, u, e, pi, alpha, costh;\n    Vectors p1, p2, p3, p4;\n    \n    * 1. Amplitude Squared (Diagrammatica Trace Rules)\n    Local Msq = (e^4 / s^2) * d_(mu, mu1) * d_(nu, nu1) * (g_(1, p2) * g_(1, mu) * g\n    _(1, p1) * g_(1, nu)) *\n        (g_(2, p3) * g_(2, mu1) * g_(2, p4) * g_(2, nu1));\n    \n    Local dSigma = (1 / (64 * pi^2 * s)) * Msq;\n    \n    trace4, 1;\n    trace4, 2;\n    .sort \n\nTime =       0.00 sec    Generated terms =          2\n             Msq         Terms in output =          2\n                         Bytes used      =        140\n\nTime =       0.00 sec    Generated terms =          2\n          dSigma         Terms in output =          2\n                         Bytes used      =        156\n    Print Msq; \n    .sort\n\nTime =       0.00 sec    Generated terms =          2\n             Msq      

In [39]:
import re
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-muted') 
plt.rcParams['axes.grid'] = True

# Capture the output of the previous FORM cell
form_out = _ 

if form_out is None:
    print("No FORM output detected. Did the previous cell run successfully?")
    # Regex looks for 'dSigma =' specifically in the output section (indented)
    # and grabs the content inside the parentheses
match = re.search(r'\n\s+dSigma\s*=\s*.*?\(\s*(.*?)\s*\)\s*;', form_out, re.DOTALL)
    
if match is None :
    print("No FORM output detected. Did the previous cell run successfully?")
        
expr_str = match.group(1)
# Convert FORM to Python: ^ -> ** and costh -> x
py_expr = expr_str.replace('^', '**').replace('costh', 'x')
        
# Physical Constants
alpha = 1/137.036
s_val = 100**2  # Center of mass energy squared
        
# Prepare data
f_ang = lambda x: eval(py_expr)
cos_theta = np.linspace(-1, 1, 100)
        
# s^-1 * alpha^2 is the prefactor from your FORM output
y_vals = (alpha**2 / s_val) * f_ang(cos_theta)
        
plt.figure(figsize=(8, 5))
plt.plot(cos_theta, y_vals, lw=2, label=r'$e^+e^- \to \mu^+\mu^-$')
plt.title("Differential Cross Section (Massless Limit)")
plt.xlabel(r"$\cos(\theta)$")
plt.ylabel(r"$d\sigma/d\Omega$")
plt.legend()
plt.show()
    


AttributeError: 'RcParams' object has no attribute '_get'

In [32]:
!form ../scripts/ee_to_mumu.frm

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Wed Apr  1 16:45:21 2026
    * Process: e+ e- -> mu+ mu-
    Indices mu, nu, mu1, nu1;
    Symbols s, t, u, e, pi, alpha, costh;
    Vectors p1, p2, p3, p4;
    
    * 1. Amplitude Squared (Diagrammatica Trace Rules)
    Local Msq = (e^4 / s^2) * d_(mu, mu1) * d_(nu, nu1) * (g_(1, p2) * g_(1, mu) * g
    _(1, p1) * g_(1, nu)) *
        (g_(2, p3) * g_(2, mu1) * g_(2, p4) * g_(2, nu1));
    
    Local dSigma = (1 / (64 * pi^2 * s)) * Msq;
    
    trace4, 1;
    trace4, 2;
    .sort 

Time =       0.00 sec    Generated terms =          2
             Msq         Terms in output =          2
                         Bytes used      =        140

Time =       0.00 sec    Generated terms =          2
          dSigma         Terms in output =          2
                         Bytes used      =        156
    Print Msq; 
    .sort

Time =       0.00 sec    Generated terms =          2
             Msq         Terms in output =         